In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv


In [ ]:
%%writefile /kaggle/working/my_agent.py
# --- BaseAgent Definition ---
from __future__ import annotations

from abc import ABC, abstractmethod
from collections.abc import Sequence

from arcengine import FrameData, GameAction, GameState


class BaseAgent(ABC):
    """Abstract base class for all ARC-AGI-3 agents."""

    @abstractmethod
    def choose_action(
        self,
        frames: Sequence[FrameData],
        latest_frame: FrameData,
    ) -> GameAction | tuple[GameAction, tuple[int, int]]:
        """Choose the next action given frame history and latest frame."""

    def is_done(
        self,
        frames: Sequence[FrameData],
        latest_frame: FrameData,
    ) -> bool:
        """Return True if agent should terminate play for this environment."""
        return latest_frame.state in (GameState.WIN, GameState.GAME_OVER)

    def reset(self) -> None:
        """Reset internal episodic state between games."""


# --- Agent Implementation ---
from __future__ import annotations

import random
from collections.abc import Sequence

import numpy as np
import torch
import torch.nn.functional as F
from arcengine import FrameData, GameAction
from torch import nn




class PolicyCNN(nn.Module):
    """Compact ConvNet mapping grid observations (16 colors) to action distributions."""

    def __init__(self, num_actions: int = 7) -> None:
        super().__init__()
        # Input: 16 channels (one-hot color states 0-15) over 64x64 grid
        self.conv1 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)  # 32x32
        self.conv3 = nn.Conv2d(64, 64, kernel_size=3, stride=2, padding=1)  # 16x16
        self.pool = nn.AdaptiveAvgPool2d((4, 4))
        self.fc = nn.Linear(64 * 4 * 4, num_actions)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = F.relu(self.conv1(x))
        h = F.relu(self.conv2(h))
        h = F.relu(self.conv3(h))
        h = self.pool(h)
        h = h.view(h.size(0), -1)
        return self.fc(h)


class StochasticGoose(BaseAgent):
    """Tufa Labs Milestone 1 CNN action learner."""

    def __init__(self, epsilon: float = 0.2, seed: int = 42) -> None:
        self.rng = random.Random(seed)
        self.epsilon = epsilon
        self.device = torch.device(
            "mps"
            if torch.backends.mps.is_available()
            else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        self.policy = PolicyCNN(num_actions=7).to(self.device)
        self.policy.eval()

    def _encode_grid(self, frame: FrameData) -> torch.Tensor:
        grid = np.zeros((64, 64), dtype=np.int64)
        if frame.frame and len(frame.frame) > 0:
            raw = frame.frame[0]
            if isinstance(raw, list) and len(raw) > 0 and isinstance(raw[0], list):
                h = min(len(raw), 64)
                w = min(len(raw[0]), 64)
                grid[:h, :w] = np.array(raw)[:h, :w]

        # One-hot encode 16 color channels: shape (1, 16, 64, 64)
        tensor_grid = torch.from_numpy(grid).long()
        one_hot = F.one_hot(tensor_grid, num_classes=16).permute(2, 0, 1).unsqueeze(0).float()
        return one_hot.to(self.device)

    def choose_action(
        self,
        frames: Sequence[FrameData],
        latest_frame: FrameData,
    ) -> GameAction:
        available = latest_frame.available_actions
        if not available:
            return GameAction.RESET

        # Epsilon-greedy exploration
        if self.rng.random() < self.epsilon:
            action_int = self.rng.choice(available)
        else:
            with torch.no_grad():
                x = self._encode_grid(latest_frame)
                logits = self.policy(x).squeeze(0).cpu().numpy()

                # Mask unavailable actions
                masked_logits = np.full_like(logits, -1e9)
                for act_id in available:
                    # arcengine action integers 1-7 map to 0-6
                    idx = act_id - 1
                    if 0 <= idx < len(logits):
                        masked_logits[idx] = logits[idx]

                action_int = int(np.argmax(masked_logits)) + 1

        action = GameAction.from_id(action_int)
        if action == GameAction.ACTION6:
            x = self.rng.randint(0, 63)
            y = self.rng.randint(0, 63)
            action.set_data({"x": x, "y": y})

        return action

# --- Official Framework Adapter ---

try:
    from ..agent import Agent as OfficialAgent
except (ImportError, ValueError):
    try:
        from agents.agent import Agent as OfficialAgent
    except (ImportError, ValueError):
        OfficialAgent = object


class MyAgent(OfficialAgent):
    """Adapter wrapping StochasticGoose for official ARC-AGI-3-Agents framework."""
    MAX_ACTIONS = 400

    def __init__(self, *args, **kwargs):
        if OfficialAgent is not object:
            super().__init__(*args, **kwargs)
        self.inner_agent = StochasticGoose()

    def choose_action(self, frames, latest_frame):
        return self.inner_agent.choose_action(frames, latest_frame)

    def is_done(self, frames, latest_frame):
        return self.inner_agent.is_done(frames, latest_frame)

    def reset(self):
        self.inner_agent.reset()



In [ ]:
import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Wait for gateway to be ready
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy official framework to writable location
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Copy custom agent into templates
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Minimal init with custom agent registration
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write('''from typing import Type
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
''')

    # Environment overrides for gateway
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write('''SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=competition-key
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
''')

    # Run agent against all 110 games
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent


In [ ]:
import os
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    # Interactive / non-rerun commit mode dummy fallback
    import pandas as pd
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print("Dummy submission saved successfully for commit validation.")
